# Tamil caste & migration hate speech — 3 transformers on Kaggle (one model per cell)
**Models:** TamilSBERT, MuRIL, BLOOM-560M. Run the cells **top to bottom, one at a time**.
Each finished model saves `results/<tag>.json` + logits, so if the session dies just re-run cells 1–4 and the finished models are **skipped automatically**.

**Kaggle setup (once)**
1. Upload your zip: *Add data → Upload → New dataset* (Kaggle unzips it). Add it to this notebook.
2. Right panel → *Session options*: **Accelerator = GPU T4 x2** (we use only 1 GPU), **Internet = On** (needs phone-verified account; required to download models).
3. Run the cells below.

## Cell 1 — install + GPU setup

In [ ]:
!pip install -q peft
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"       # use ONE GPU only (T4 x2 would otherwise split batches across both)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import torch
print("GPU available:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## Cell 2 — load & clean data (same preprocessing as Part A)

In [ ]:
import glob, re, unicodedata, json, time, gc, shutil, math, dataclasses
import numpy as np, pandas as pd

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True) + glob.glob(f"./**/{name}", recursive=True)
    assert hits, f"{name} not found - did you add the dataset to the notebook?"
    return hits[0]
train, dev, test = (pd.read_csv(find(f"{n}.csv")) for n in ["train", "dev", "test"])

def clean_text(t):
    t = unicodedata.normalize("NFC", str(t))
    t = re.sub(r"[\u200b\u200c\u200d\ufeff]", "", t)
    t = t.replace("\\n", " ")
    t = re.sub(r"https?://\S+|www\.\S+", " ", t)
    t = re.sub(r"@\S+", " ", t)
    t = re.sub(r"\b\d{1,2}:\d{2}(:\d{2})?\b", " ", t)
    t = re.sub(r"(.)\1{2,}", r"\1\1", t)
    return re.sub(r"\s+", " ", t.lower()).strip()

def build(df):
    df = df.copy(); df["clean"] = df.text.apply(clean_text); return df[df.clean.str.len() > 0]
tr, dv, te = build(train), build(dev), build(test)
conf = tr.groupby("clean").label.nunique(); conf = conf[conf > 1].index
tr = tr[~tr.clean.isin(conf)].drop_duplicates("clean").reset_index(drop=True)
dv = dv[~dv.clean.isin(tr.clean)].drop_duplicates("clean").reset_index(drop=True)

from sklearn.model_selection import train_test_split
tr_fit, tr_val = train_test_split(tr, test_size=0.1, stratify=tr.label, random_state=42)  # val slice picks best epoch/lr
CW = torch.tensor(len(tr_fit) / (2 * np.bincount(tr_fit.label)), dtype=torch.float)       # balanced class weights
print("fit", len(tr_fit), "| val", len(tr_val), "| dev (final score only)", len(dv), "| test", len(te), "| class weights", CW.tolist())

## Cell 3 — helper functions (just run, nothing trains yet)

In [ ]:
from torch.utils.data import Dataset
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, EarlyStoppingCallback, set_seed)
OUT = "results"; os.makedirs(OUT, exist_ok=True)
CKPT = "/kaggle/temp/ckpt" if os.path.exists("/kaggle/temp") else "ckpt"

class TextDS(Dataset):
    def __init__(self, texts, labels, tok, max_len=128):
        self.enc = tok(list(texts), truncation=True, max_length=max_len)
        self.labels = None if labels is None else list(labels)
    def __len__(self): return len(self.enc["input_ids"])
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        if self.labels is not None: item["labels"] = int(self.labels[i])
        return item

def compute_metrics(p):
    logits, y = p; pr = logits.argmax(-1)
    return {"accuracy": accuracy_score(y, pr), "macro_f1": f1_score(y, pr, average="macro")}

class WeightedTrainer(Trainer):
    def __init__(self, *a, class_weights=None, **k):
        super().__init__(*a, **k); self.cw = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels"); out = model(**inputs)
        w = None if self.cw is None else self.cw.to(out.logits.device)
        loss = torch.nn.functional.cross_entropy(out.logits.float(), labels, weight=w)
        return (loss, out) if return_outputs else loss

def run_model(tag, model_name, force=False, seed=42, epochs=5, lr=2e-5, bs=16, max_len=128,
              patience=2, lora=False, fp16=True):
    """Train ONE model, save metrics+logits to results/<tag>.*, free the GPU. Skips if already done."""
    res_path = f"{OUT}/{tag}.json"
    if os.path.exists(res_path) and not force:
        print("already done, skipping:", tag); return json.load(open(res_path))
    set_seed(seed); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); t0 = time.time()

    tok = AutoTokenizer.from_pretrained(model_name)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    model.config.pad_token_id = tok.pad_token_id; model.config.use_cache = False
    if lora:   # train small adapters only (stable + low memory for big decoder models)
        from peft import LoraConfig, get_peft_model, TaskType
        model = get_peft_model(model, LoraConfig(task_type=TaskType.SEQ_CLS, r=16, lora_alpha=32,
                               lora_dropout=0.1, target_modules=["query_key_value"]))
        model.print_trainable_parameters()

    ds_fit, ds_val, ds_dv = (TextDS(d.clean, d.label, tok, max_len) for d in (tr_fit, tr_val, dv))
    out = f"{CKPT}/{tag}"
    valid = {f.name for f in dataclasses.fields(TrainingArguments)}
    steps = math.ceil(len(ds_fit) / bs) * epochs
    ta = dict(output_dir=out, num_train_epochs=epochs, learning_rate=lr, per_device_train_batch_size=bs,
              per_device_eval_batch_size=64, weight_decay=0.01, warmup_steps=int(0.1 * steps), max_grad_norm=1.0,
              save_strategy="epoch", load_best_model_at_end=True, metric_for_best_model="macro_f1",
              save_total_limit=1, fp16=fp16 and torch.cuda.is_available(), seed=seed, report_to="none",
              label_names=["labels"], dataloader_num_workers=2)
    ta["eval_strategy" if "eval_strategy" in valid else "evaluation_strategy"] = "epoch"
    args = TrainingArguments(**{k: v for k, v in ta.items() if k in valid})
    kw = dict(model=model, args=args, train_dataset=ds_fit, eval_dataset=ds_val, compute_metrics=compute_metrics,
              class_weights=CW, callbacks=[EarlyStoppingCallback(early_stopping_patience=patience)])
    try:    trainer = WeightedTrainer(processing_class=tok, **kw)
    except TypeError: trainer = WeightedTrainer(tokenizer=tok, **kw)
    trainer.train()

    val_f1 = trainer.evaluate()["eval_macro_f1"]
    dl = trainer.predict(ds_dv).predictions
    tl = trainer.predict(TextDS(te.clean, None, tok, max_len)).predictions
    pred = dl.argmax(-1)
    p, r, f, _ = precision_recall_fscore_support(dv.label, pred, average="macro", zero_division=0)
    res = dict(tag=tag, model=model_name, lr=lr, seed=seed, val_F1=float(val_f1), accuracy=float(accuracy_score(dv.label, pred)),
               macro_P=float(p), macro_R=float(r), macro_F1=float(f),
               weighted_F1=float(f1_score(dv.label, pred, average="weighted")),
               hate_F1=float(f1_score(dv.label, pred, pos_label=1)), predicts_both_classes=bool(len(set(pred)) == 2),
               train_minutes=round((time.time() - t0) / 60, 1),
               peak_gpu_gb=round(torch.cuda.max_memory_allocated() / 1e9, 2))
    json.dump(res, open(res_path, "w"), indent=1)
    np.savez(f"{OUT}/{tag}_logits.npz", dev=dl, test=tl)
    print(json.dumps(res, indent=1))
    if not res["predicts_both_classes"]: print("WARNING: model collapsed (predicts one class only) - don't report this run")

    del trainer, model; gc.collect(); torch.cuda.empty_cache(); shutil.rmtree(out, ignore_errors=True)   # free GPU + disk
    return res

## Cell 4 — Model 1: TamilSBERT  (~10 min)

In [ ]:
r1 = run_model("tamilsbert", "l3cube-pune/tamil-sentence-bert-nli", lr=2e-5, epochs=5, patience=2)

## Cell 5 — Model 2: MuRIL  (2 learning rates, ~15 min each)
MuRIL collapsed at lr=2e-5 on Colab, so we try 3e-5 and 1e-5 and pick the best on the *validation slice*. Run the second line only if you have time/quota.

In [ ]:
r2a = run_model("muril_lr3e-5", "google/muril-base-cased", lr=3e-5, epochs=6, patience=3)
r2b = run_model("muril_lr1e-5", "google/muril-base-cased", lr=1e-5, epochs=6, patience=3)

## Cell 6 — Model 3: BLOOM-560M with LoRA  (~20–30 min)
fp16 is OFF for BLOOM (fp16 can give NaN). If you get an out-of-memory error, change `bs=16` to `bs=8`.

In [ ]:
r3 = run_model("bloom560m_lora", "bigscience/bloom-560m", lr=2e-4, epochs=4, patience=2, lora=True, fp16=False)

## Cell 7 — compare, ensemble, save test predictions, zip for download

In [ ]:
import glob
R = pd.DataFrame([json.load(open(f)) for f in sorted(glob.glob(f"{OUT}/*.json"))])
cols = ["tag", "val_F1", "macro_F1", "macro_P", "macro_R", "accuracy", "hate_F1", "predicts_both_classes", "train_minutes", "peak_gpu_gb"]
print(R[cols].round(4).to_string(index=False))

ok = R[R.predicts_both_classes].copy()
best = ok.sort_values("val_F1", ascending=False).drop_duplicates("model")     # best config per model, chosen on validation
print("\nBest per model (chosen on validation):"); print(best[["model", "tag", "val_F1", "macro_F1"]].round(4).to_string(index=False))

from scipy.special import softmax
L = {t: np.load(f"{OUT}/{t}_logits.npz") for t in best.tag}
if len(L) > 1:
    ens_dev = np.mean([softmax(v["dev"], axis=1) for v in L.values()], axis=0)
    print("Ensemble macro-F1 on dev:", round(f1_score(dv.label, ens_dev.argmax(1), average="macro"), 4))

top = best.sort_values("val_F1", ascending=False).iloc[0].tag
pd.DataFrame({"id": te.id, "label": L[top]["test"].argmax(1)}).to_csv(f"{OUT}/test_predictions.csv", index=False)
best.drop(columns=["tag"]).round(4).to_csv(f"{OUT}/final_transformers.csv", index=False)
shutil.make_archive("results_download", "zip", OUT)
print("Test predictions from:", top, "| download results_download.zip from the Output tab")